# 05 - Previsões com o modelo vigente e teste integrado

**Etapa 8** | Fluxo: passo 10

1. Carrega o **modelo vigente** pelo manifesto: o pipeline serializado já traz o pré-processamento e as features eleitas.
2. Gera previsões para **registros novos, sem rótulo** (ids fora de treino/teste, nunca usados em nenhuma decisão).
3. **Teste integrado** do ciclo: a votação foi refeita? os dois algoritmos foram treinados? o vigente foi promovido ou mantido
   automaticamente? O notebook falha (e o workflow para) se alguma verificação não passar.
4. Compara o ciclo atual com o anterior (o que mudou) e consulta o histórico no MLflow.

In [1]:
# Preparação: localiza a raiz do projeto (o nbconvert executa com a pasta notebooks/ como diretório atual)
import sys, json, warnings
from pathlib import Path
RAIZ = Path.cwd().resolve()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow

from src.config import carregar_config, p, dir_, ler_json, gravar_json
from src import ciclo, rastreio

cfg = carregar_config()
pd.set_option("display.width", 160, "display.max_columns", 30)

In [2]:
from src import features as ft, modelagem as mo, promocao as pr, preprocessamento as pre
ctx = ciclo.carregar_contexto(cfg)
ev = ciclo.dir_evidencias(ctx)

manifesto = pr.ler_manifesto()
bundle = mo.carregar_modelo(p(manifesto["caminho_modelo"]))
pipe = bundle["pipeline"]
assert list(bundle["features"]) == list(manifesto["features"]), "features do modelo != manifesto"
print("Modelo vigente:", manifesto["modelo_id"], "| algoritmo:", manifesto["algoritmo"], "| treinado em", manifesto["data_version"])

novos = pd.read_csv(p(ctx["scoring_path"]))
print("registros a pontuar:", len(novos), "| colunas com rótulo?", cfg["dados"]["alvo"] in novos.columns)

Modelo vigente: C007_20261127T121405Z_random_forest | algoritmo: random_forest | treinado em v007_4f092bbb0e
registros a pontuar: 152 | colunas com rótulo? False


In [3]:
# Mesma preparação do ETL: regras de domínio + atributos derivados; o pipeline cuida do resto (imputação, escala, one-hot, features)
prep = ft.criar_atributos(pre.aplicar_regras_de_dominio(novos, cfg))
proba = pipe.predict_proba(prep[ft.COLS_NUM + ft.COLS_CAT])[:, 1]
saida = pd.DataFrame({
    cfg["dados"]["coluna_id"]: novos[cfg["dados"]["coluna_id"]], "prob_churn": proba.round(4),
    "previsao_churn": (proba >= 0.5).astype(int), "modelo_id": manifesto["modelo_id"], "cycle_id": ctx["cycle_id"]})
saida.to_csv(ev / "previsoes.csv", index=False)
print(f"previsões: {len(saida)} | churn previsto: {saida['previsao_churn'].mean():.1%} | prob. média: {saida['prob_churn'].mean():.3f}")
saida.sort_values("prob_churn", ascending=False).head(8)

previsões: 152 | churn previsto: 55.3% | prob. média: 0.539


,id_cliente,prob_churn,previsao_churn,modelo_id,cycle_id
40,9009040,0.9833,1,C007_20261127T121405Z_random_forest,C010_20261128T152939Z
149,9009149,0.9720,1,C007_20261127T121405Z_random_forest,C010_20261128T152939Z
87,9009087,0.9629,1,C007_20261127T121405Z_random_forest,C010_20261128T152939Z
131,9009131,0.9526,1,C007_20261127T121405Z_random_forest,C010_20261128T152939Z
128,9009128,0.9422,1,C007_20261127T121405Z_random_forest,C010_20261128T152939Z
118,9009118,0.9281,1,C007_20261127T121405Z_random_forest,C010_20261128T152939Z
138,9009138,0.9223,1,C007_20261127T121405Z_random_forest,C010_20261128T152939Z
93,9009093,0.9085,1,C007_20261127T121405Z_random_forest,C010_20261128T152939Z


## Teste integrado do ciclo

In [4]:
rastreio.configurar(cfg)
runs = rastreio.historico_runs(cfg)
do_ciclo = runs[runs["tags.cycle_id"] == ctx["cycle_id"]]
por_etapa = do_ciclo["tags.etapa"].value_counts().to_dict()

treino_ids = set(pd.read_csv(dir_(cfg, "dir_processed") / "treino.csv")[cfg["dados"]["coluna_id"]])
teste_ids = set(pd.read_csv(dir_(cfg, "dir_processed") / "teste.csv")[cfg["dados"]["coluna_id"]])
modelos = do_ciclo[do_ciclo["tags.etapa"] == "treino_modelo"]
decisao_run = do_ciclo[do_ciclo["tags.etapa"] == "decisao"]
votacao_run = do_ciclo[do_ciclo["tags.etapa"] == "votacao"]

verificacoes = {
    "votação refeita neste ciclo (1 run 'votacao')": por_etapa.get("votacao", 0) == 1,
    "dois algoritmos treinados (2 runs 'treino_modelo')": por_etapa.get("treino_modelo", 0) == 2
        and set(modelos["tags.candidato"]) == {"A", "B"},
    "A e B com as mesmas features eleitas": modelos["tags.features_eleitas"].nunique() == 1
        and modelos["tags.features_eleitas"].iloc[0] == votacao_run["tags.features_eleitas"].iloc[0],
    "mesma data_version em todas as runs do ciclo": do_ciclo["tags.data_version"].nunique() == 1,
    "decisão automática registrada (promovido/mantido)": len(decisao_run) == 1
        and decisao_run["tags.decisao"].iloc[0] in ("promovido", "mantido"),
    "manifesto do vigente referencia este ciclo": manifesto["ultima_avaliacao"]["cycle_id"] == ctx["cycle_id"],
    "previsões sem ausentes e uma por registro": saida["prob_churn"].notna().all() and len(saida) == len(novos),
    "registros pontuados fora de treino e teste": set(novos[cfg["dados"]["coluna_id"]]).isdisjoint(treino_ids | teste_ids),
    "evidências do ciclo gravadas (decisao.json)": (ev / "decisao.json").exists(),
}
tab_ver = pd.DataFrame({"verificação": list(verificacoes), "ok": list(verificacoes.values())})
print(tab_ver.to_string(index=False))
assert all(verificacoes.values()), "teste integrado FALHOU - veja a tabela acima"
print("\nTESTE INTEGRADO: OK")

                                       verificação   ok
     votação refeita neste ciclo (1 run 'votacao') True
dois algoritmos treinados (2 runs 'treino_modelo') True
              A e B com as mesmas features eleitas True
      mesma data_version em todas as runs do ciclo True
 decisão automática registrada (promovido/mantido) True
        manifesto do vigente referencia este ciclo True
         previsões sem ausentes e uma por registro True
        registros pontuados fora de treino e teste True
       evidências do ciclo gravadas (decisao.json) True

TESTE INTEGRADO: OK


## O que mudou entre ciclos

In [5]:
hist = ciclo.ler_historico()
print("Ciclos no histórico:", len(hist))
display(hist[["cycle_id", "gatilho", "data_version", "n_lotes", "n_linhas", "vencedor", "f1_cv_A", "f1_cv_B",
              "f1_teste_vencedor", "f1_teste_vigente_anterior", "decisao", "modelo_vigente_id", "drift_alerta"]])
if len(hist) >= 2:
    a, b = hist.iloc[-2], hist.iloc[-1]
    fa, fb = set(str(a["features_eleitas"]).split("|")), set(str(b["features_eleitas"]).split("|"))
    mudancas = pd.DataFrame({
        "ciclo anterior": [a["cycle_id"], a["data_version"], a["n_linhas"], a["vencedor"], a["decisao"], a["modelo_vigente_id"]],
        "ciclo atual": [b["cycle_id"], b["data_version"], b["n_linhas"], b["vencedor"], b["decisao"], b["modelo_vigente_id"]]},
        index=["cycle_id", "data_version", "linhas", "vencedor do ciclo", "decisão", "modelo vigente"])
    display(mudancas)
    print("features que ENTRARAM:", sorted(fb - fa) or "-")
    print("features que SAÍRAM  :", sorted(fa - fb) or "-")

Ciclos no histórico: 10


,cycle_id,gatilho,data_version,n_lotes,n_linhas,vencedor,f1_cv_A,f1_cv_B,f1_teste_vencedor,f1_teste_vigente_anterior,decisao,modelo_vigente_id,drift_alerta
0,C001_20261124T234632Z,workflow_dispatch,v001_8c382cd5af,1,3000,modelo_B,0.6717,0.6792,0.6804,NaN,promovido,C001_20261124T234632Z_random_forest,False
1,C002_20261125T083010Z,schedule,v002_e6352cb52b,2,3800,modelo_B,0.6656,0.6690,0.6862,0.6777,promovido,C002_20261125T083010Z_random_forest,False
2,C003_20261125T191557Z,schedule,v003_d73cde0298,3,4600,modelo_A,0.6666,0.6679,0.6757,0.6913,mantido,C002_20261125T083010Z_random_forest,False
3,C004_20261126T033330Z,schedule,v004_58bb9be60f,4,5400,modelo_B,0.6707,0.6742,0.7135,0.7046,promovido,C004_20261126T033330Z_random_forest,True
4,C005_20261126T165209Z,schedule,v005_1b63121e73,5,6200,modelo_A,0.6798,0.6773,0.7080,0.7114,mantido,C004_20261126T033330Z_random_forest,True
5,C006_20261127T014450Z,schedule,v006_0cb9f395e1,6,7000,modelo_A,0.6940,0.6947,0.7125,0.7101,mantido,C004_20261126T033330Z_random_forest,True
6,C007_20261127T121405Z,schedule,v007_4f092bbb0e,7,7800,modelo_B,0.6953,0.7017,0.7258,0.7099,promovido,C007_20261127T121405Z_random_forest,True
7,C008_20261127T211728Z,schedule,v008_e9edeb160c,8,8600,modelo_A,0.7139,0.7139,0.7130,0.7271,mantido,C007_20261127T121405Z_random_forest,True
8,C009_20261128T032133Z,schedule,v009_0492196107,9,9400,modelo_B,0.7165,0.7194,0.7196,0.7275,mantido,C007_20261127T121405Z_random_forest,True
9,C010_20261128T152939Z,schedule,v010_bdc553b068,10,10200,modelo_A,0.7210,0.7223,0.7302,0.7363,mantido,C007_20261127T121405Z_random_forest,True


,ciclo anterior,ciclo atual
cycle_id,C009_20261128T032133Z,C010_20261128T152939Z
data_version,v009_0492196107,v010_bdc553b068
linhas,9400,10200
vencedor do ciclo,modelo_B,modelo_A
decisão,mantido,mantido
modelo vigente,C007_20261127T121405Z_random_forest,C007_20261127T121405Z_random_forest


features que ENTRARAM: ['num__uso_dados_gb', 'num__uso_por_servico']
features que SAÍRAM  : ['cat__plano_premium', 'num__idade']


In [6]:
# Histórico recuperável no MLflow (várias execuções/ciclos na mesma consulta)
runs = rastreio.historico_runs(cfg)
print("runs no experimento:", len(runs), "| ciclos distintos:", runs["tags.cycle_id"].nunique())
cols = [c for c in ["tags.cycle_id", "tags.etapa", "tags.candidato", "tags.decisao", "metrics.cv_f1_macro_media",
                    "metrics.teste_vencedor", "metrics.delta_teste"] if c in runs.columns]
runs[cols].sort_values(["tags.cycle_id", "tags.etapa"]).tail(15)

runs no experimento: 59 | ciclos distintos: 10


,tags.cycle_id,tags.etapa,tags.candidato,tags.decisao,metrics.cv_f1_macro_media,metrics.teste_vencedor,metrics.delta_teste
11,C008_20261127T211728Z,previsoes,None,None,NaN,NaN,NaN
13,C008_20261127T211728Z,treino_modelo,B,None,0.713926,NaN,NaN
14,C008_20261127T211728Z,treino_modelo,A,None,0.713857,NaN,NaN
15,C008_20261127T211728Z,votacao,None,None,NaN,NaN,NaN
6,C009_20261128T032133Z,decisao,None,mantido,NaN,0.719595,-0.007883
10,C009_20261128T032133Z,etl,None,None,NaN,NaN,NaN
5,C009_20261128T032133Z,previsoes,None,None,NaN,NaN,NaN
7,C009_20261128T032133Z,treino_modelo,B,None,0.719440,NaN,NaN
8,C009_20261128T032133Z,treino_modelo,A,None,0.716496,NaN,NaN
9,C009_20261128T032133Z,votacao,None,None,NaN,NaN,NaN


In [7]:
with rastreio.run("previsoes_vigente", ctx, etapa="previsoes", tags_extra={"modelo_vigente_id": manifesto["modelo_id"]}) as r:
    mlflow.log_params({"modelo_id": manifesto["modelo_id"], "n_registros": len(saida)})
    mlflow.log_metrics({"taxa_churn_prevista": float(saida["previsao_churn"].mean()), "prob_media": float(saida["prob_churn"].mean())})
    mlflow.log_artifact(str(ev / "previsoes.csv"))
    ctx["run_ids"]["previsoes"] = r.info.run_id
ciclo.salvar_contexto(cfg, ctx)
print("Ciclo concluído:", ctx["cycle_id"])

Ciclo concluído: C010_20261128T152939Z
